<a href="https://colab.research.google.com/github/fatasa/Tesis/blob/main/augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import cv2
import numpy as np
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')

#PATH GOOGLE DRIVE
DATASET_DRIVE_PATH = '/content/drive/MyDrive/cell_images'
AUGMENTED_DRIVE_PATH = '/content/drive/MyDrive/cell_images_augmented'

Mounted at /content/drive


In [2]:
def apply_augmentation(image):
    augmented = []
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)

    # 1. Rotasi 30 derajat
    M = cv2.getRotationMatrix2D(center, 30, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    augmented.append(('rot30', rotated))

    # 2. Horizontal Flip
    h_flip = cv2.flip(image, 1)
    augmented.append(('fliph', h_flip))

    # 3. Vertical Flip
    v_flip = cv2.flip(image, 0)
    augmented.append(('flipv', v_flip))

    return augmented

In [3]:
def load_raw_data(dataset_dir):
    images = []
    labels = []
    file_names = []
    categories = ['Uninfected', 'Parasitized'] # 0: Uninfected, 1: Parasitized[cite: 1, 2]

    for category in categories:
        folder_path = os.path.join(dataset_dir, category)
        label = 0 if category == 'Uninfected' else 1

        if not os.path.exists(folder_path):
            print(f"⚠️ Warning: Folder {folder_path} tidak ditemukan!")
            continue

        print(f"Memuat citra dari folder: {category}...")
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            if img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                img = cv2.imread(img_path)
                if img is not None:
                    images.append(img)
                    labels.append(label)
                    file_names.append(img_name)

    print(f"\nTotal citra mentah berhasil dimuat: {len(images):,} gambar.")
    return images, np.array(labels), file_names

In [4]:
if __name__ == "__main__":
    X_raw, y_raw, filenames = load_raw_data(DATASET_DRIVE_PATH)

    for cat in ["Uninfected", "Parasitized"]:
        os.makedirs(os.path.join(AUGMENTED_DRIVE_PATH, cat), exist_ok=True)

    print("\n---> Menyimpan Seluruh Data Ter-augmentasi ke Google Drive...")
    for img, label, fname in tqdm(zip(X_raw, y_raw, filenames), total=len(X_raw), desc="Proses Augmentasi"):
        cat_dir = os.path.join(AUGMENTED_DRIVE_PATH, "Parasitized" if label == 1 else "Uninfected")

        clean_fname = os.path.basename(fname)
        base_name = os.path.splitext(clean_fname)[0]

        # Validasi format data piksel (uint8)
        img_save = img.astype(np.uint8) if img.dtype != np.uint8 else img

        # Simpan citra asli
        cv2.imwrite(os.path.join(cat_dir, f"{base_name}_ori.png"), img_save)

        # Simpan citra hasil augmentasi (Rotasi, H-Flip, V-Flip)
        for aug_type, aug_img in apply_augmentation(img_save):
            cv2.imwrite(os.path.join(cat_dir, f"{base_name}_{aug_type}.png"), aug_img)

    print(f"\n[SELESAI] Seluruh dataset hasil augmentasi tersimpan di:\n{AUGMENTED_DRIVE_PATH}")

Memuat citra dari folder: Uninfected...
Memuat citra dari folder: Parasitized...

Total citra mentah berhasil dimuat: 27,558 gambar.

---> Menyimpan Seluruh Data Ter-augmentasi ke Google Drive...


Proses Augmentasi: 100%|██████████| 27558/27558 [53:42<00:00,  8.55it/s]


[SELESAI] Seluruh dataset hasil augmentasi tersimpan di:
/content/drive/MyDrive/cell_images_augmented


In [5]:
    def count_files_in_dir(folder_path):
        if not os.path.exists(folder_path):
            return 0
        return len([f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    uninfected_cnt = count_files_in_dir(os.path.join(AUGMENTED_DRIVE_PATH, "Uninfected"))
    parasitized_cnt = count_files_in_dir(os.path.join(AUGMENTED_DRIVE_PATH, "Parasitized"))
    total_all = uninfected_cnt + parasitized_cnt

    print("\n" + "="*55)
    print("      REKAPITULASI TOTAL DATASET HASIL AUGMENTASI")
    print("="*55)
    print(f"   - Uninfected  : {uninfected_cnt:,} citra")
    print(f"   - Parasitized : {parasitized_cnt:,} citra")
    print("-" * 55)
    print(f"TOTAL KESELURUHAN CITRA: {total_all:,} citra")
    print("="*55)


      REKAPITULASI TOTAL DATASET HASIL AUGMENTASI
   - Uninfected  : 55,116 citra
   - Parasitized : 55,116 citra
-------------------------------------------------------
TOTAL KESELURUHAN CITRA: 110,232 citra
